### prompt와 model 연결

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자가 입력한 요리의 레시피를 생각해 주세요.."),
        ("human", "{dish}")
    ]
)

model = ChatOpenAI(model="gpt-6-luna")

In [2]:
# 그리고 이들을 연결한 chain을 생성
chain = prompt | model

# 이 chain을 실행
ai_message = chain.invoke({"dish": "카레"})
print(ai_message.content)

## 기본 카레라이스 레시피 (2~3인분)

### 재료
- 카레 블록 3~4조각
- 돼지고기 또는 닭고기 200g
- 감자 1개
- 양파 1개
- 당근 ½개
- 물 500~600ml
- 식용유 약간
- 밥 2~3공기

### 만드는 법
1. 고기와 감자, 양파, 당근을 한입 크기로 썰어요.
2. 냄비에 식용유를 두르고 고기를 볶아요. 고기 겉면이 익으면 양파, 당근, 감자를 넣어 2~3분 더 볶아요.
3. 물을 붓고 끓으면 중불에서 재료가 부드러워질 때까지 15분 정도 익혀요.
4. 불을 약하게 줄이고 카레 블록을 넣어 잘 풀어요. 눌어붙지 않도록 저으면서 5분 정도 더 끓여요.
5. 밥에 카레를 얹으면 완성이에요.

카레가 너무 걸쭉하면 물을 조금 더 넣고, 묽으면 약불에서 더 끓여 농도를 맞추세요.


### StrOutParser를 연결에 추가

In [3]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()
output = chain.invoke({"dish": "카레"})
print(output)

### 기본 카레 레시피 (3~4인분)

**재료**
- 카레 가루 또는 고형 카레 1팩
- 밥 3~4공기
- 감자 2개
- 양파 1개
- 당근 ½개
- 돼지고기나 닭고기 200g
- 물 600~700ml
- 식용유 약간

**만드는 법**
1. 감자, 양파, 당근, 고기를 먹기 좋은 크기로 썰어요.
2. 냄비에 식용유를 두르고 고기를 볶다가, 양파와 당근을 넣어 함께 볶아요.
3. 감자를 넣고 물을 부은 뒤, 감자가 익을 때까지 15~20분 끓여요.
4. 불을 약하게 줄이고 카레를 조금씩 넣어 잘 풀어요. 걸쭉해질 때까지 저어가며 5분 정도 더 끓여요.
5. 밥 위에 카레를 얹어 내면 완성입니다.

양파를 충분히 볶으면 카레가 더 달고 깊은 맛이 나요.


### PydanticOutputParser를 사용한 연결

In [4]:
# 먼저 Recipe 클래스를 정의하고, output_parser(PydanticOutputParser)를 준비
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")

output_parser = PydanticOutputParser(pydantic_object=Recipe)

In [ ]:
# 이어서 prompt(PromptTemplate)와 model(ChatOpenAI)을 준비
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate(
    [
        (
            "system",
            "사용자가 입력한 요리의 레시피를 생각해 주세요.\n\n{format_instructions}"
        ),
        ("human", "{dish}")
    ]
)

prompt_with_format_instructions = prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

model = ChatOpenAI(model="gpt-6-luna").bind(response_format={"type": "json_object"})

In [6]:
# LCEL 표기법으로 prompt와 model.output_parser를 연결한 chain을 생성
chain = prompt_with_format_instructions | model | output_parser

# chain을 실행해 보겠다.

recipe = chain.invoke({"dish": "카레"})
print(type(recipe))
print(recipe)

<class '__main__.Recipe'>
ingredients=['카레 가루 또는 고형 카레 100g', '돼지고기 또는 닭고기 200g', '감자 2개', '당근 1개', '양파 1개', '물 700ml', '식용유 1큰술', '소금과 후추 약간', '밥 2~3공기'] steps=['고기와 감자, 당근, 양파를 한입 크기로 썹니다.', '냄비에 식용유를 두르고 양파를 중불에서 볶습니다.', '고기를 넣고 겉면이 익을 때까지 볶은 뒤 감자와 당근을 넣어 함께 볶습니다.', '물을 붓고 끓인 다음, 재료가 부드러워질 때까지 중약불에서 15~20분 끓입니다.', '불을 약하게 줄이고 카레를 넣어 잘 풀어 줍니다. 바닥에 눌어붙지 않도록 저으면서 5분 정도 더 끓입니다.', '소금과 후추로 간을 맞추고 밥 위에 카레를 담아 냅니다.']
